# VeriCall — 5-Minute End-to-End Pipeline Smoke Test
### Ultra-Fast Verification on Kaggle GPU (T4 / P100 / L4)
**Author:** Bhuvnesh Sahu ([@bhuvneshsahu01](https://github.com/bhuvneshsahu01))

This notebook executes a **rapid, lightweight dry run** of all 10 stages of the VeriCall pipeline:
1. **Dependency & Repo Detection:** Clones/pulls the repo into `/kaggle/working/vericall`.
2. **16-Endpoint Sandbox:** Verifies all 16 Pydantic v2 schemas across 5 domains.
3. **Model Loading:** Loads `Qwen/Qwen2.5-1.5B-Instruct` (or `7B-Instruct`) in 4-bit.
4. **Base Model Evaluation:** Evaluates 25 unseen API requests with candidate schemas.
5. **Fast SFT Training:** 50 samples, 15 steps with completion-only loss masking.
6. **SFT Evaluation:** Evaluates the 25 unseen API requests on SFT.
7. **DPO Mining:** Loads 25 execution-grounded near-miss preference pairs.
8. **Fast DPO Alignment:** 10 steps with anti-collapse safeguards ($\beta=0.1$, $\text{lr}=5\times 10^{-6}$, $\text{rpo\_alpha}=0.2$).
9. **Final Evaluation:** Evaluates DPO model, computes Bootstrap CI and McNemar tests.
10. **Artifact Export:** Saves `/kaggle/working/smoke_eval_summary.json`.

**Total Runtime:** ~3 to 5 minutes on free Kaggle T4 GPU.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install dependencies with resilient TRL versioning
import os
import sys
from pathlib import Path

print("[*] Installing and verifying dependencies...")
!pip install -q "fastapi>=0.100.0" "pydantic>=2.0.0" "httpx>=0.24.0" "statsmodels>=0.14.0" "scipy>=1.10.0" "tabulate>=0.9.0"
!pip install -q "transformers>=4.40.0" "peft>=0.9.0" "accelerate>=0.28.0" "bitsandbytes>=0.43.0"
!pip install -q "trl<0.12.0" || pip install -q trl

# Verify and set working directory to repository root (Kaggle / Colab / Local)
if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest repository into {repo_dir}...")
        os.system(f"cd {repo_dir} && (git pull origin master || git pull origin main)")
    os.chdir(repo_dir)
elif os.path.exists("/content"):
    repo_dir = "/content/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest repository into {repo_dir}...")
        os.system(f"cd {repo_dir} && (git pull origin master || git pull origin main)")
    os.chdir(repo_dir)
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"[✓] Environment ready. Working directory: {os.getcwd()}")


### Stage 2: In-Process FastAPI Mock Sandbox & ExecutionValidator (16 Endpoints)


In [ ]:
from vericall.sandbox.validator import ExecutionValidator
from vericall.sandbox.app import app, ENDPOINT_REGISTRY, get_tool_json_schema

validator = ExecutionValidator()
print(f"[✓] Initialized ExecutionValidator with {len(validator.registry)} endpoints across 5 domains.")

# Smoke test all 16 endpoints
test_endpoints = [
    ("issue_refund", {"transaction_id": "txn_12345678", "amount": 49.99, "currency": "USD", "reason": "CUSTOMER_RETURN", "metadata": {"rep": "rep_1"}}),
    ("create_invoice", {"client_id": "cli_1001", "issue_date": "2026-11-01", "due_date": "2026-11-15", "line_items": [{"description": "Dev", "unit_price": 100.0, "quantity": 1}], "currency": "USD"}),
    ("transfer_funds", {"source_account": "acc_100001", "destination_account": "acc_100002", "amount": 250.0, "transfer_method": "ACH", "memo": "Payment"}),
    ("audit_ledger_entries", {"ledger_id": "ledg_100001", "start_date": "2026-10-01", "end_date": "2026-10-31", "entry_type": "CREDIT", "filter_flags": {"reconciled_only": True}}),
    ("query_inventory", {"sku_list": ["SKU-PROD01"], "warehouse_id": "wh_us-01", "include_out_of_stock": False, "fulfillment_channel": "WAREHOUSE_DIRECT"}),
    ("apply_discount_code", {"cart_id": "cart_100001", "promo_code": "SAVE20", "channel": "ONLINE_WEB", "customer_loyalty_pin": 1234}),
    ("update_subscription", {"subscription_id": "sub_10000001", "plan_tier": "PRO_MONTHLY", "auto_renew": True, "addon_features": ["logs"], "effective_date": "2026-12-01"}),
    ("create_support_ticket", {"customer_id": "cust_10001", "priority": "HIGH", "department": "TECHNICAL", "subject": "Platform latency issue", "ticket_context": {"origin": "web"}}),
    ("update_customer_tier", {"customer_id": "cust_10001", "target_tier": "GOLD", "assigned_rep_id": "rep_1001", "override_reason": "High spending account", "benefits": ["waived_fees"]}),
    ("schedule_webhook", {"destination_url": "https://api.partner.com/events", "subscribed_events": ["order.created"], "retry_count": 3, "signing_secret": "whsec_12345678", "headers": {"X-App": "test"}}),
    ("provision_compute_node", {"instance_tier": "c6i.xlarge", "cloud_region": "us-east-1", "cluster_count": 2, "environment_tag": "PRODUCTION", "resource_tags": {"team": "infra"}}),
    ("scale_kubernetes_cluster", {"cluster_name": "k8s-prod-101", "node_pool": "worker-pool", "desired_replicas": 5, "scaling_strategy": "SURGE_FIRST", "autoscale_config": {"min_nodes": 2, "max_nodes": 20}}),
    ("rotate_iam_credentials", {"service_account_id": "sa-data1001", "credential_type": "OAUTH2_KEY", "grace_period_hours": 48, "notify_emails": ["ops@company.com"], "force_revocation": False}),
    ("search_multi_leg_flights", {"itinerary_segments": [{"departure_airport": "JFK", "arrival_airport": "LHR", "flight_date": "2026-11-10"}], "cabin_class": "BUSINESS", "passenger_count": 1, "flexible_dates": False, "preferred_alliances": ["STAR_ALLIANCE"]}),
    ("reserve_corporate_hotel", {"property_code": "HTL-NY-1001", "check_in_date": "2026-12-01", "check_out_date": "2026-12-05", "room_category": "EXECUTIVE_SUITE", "corporate_billing_code": "CORP-10001", "guest_manifest": [{"first_name": "Jane", "last_name": "Doe"}]}),
    ("cancel_itinerary_segment", {"booking_reference": "PNR-ABC123", "segment_id": "seg_01", "cancellation_reason": "CLIENT_RESCHEDULE", "request_credit_voucher": True, "refund_override_notes": "Authorized"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call({"name": fn, "arguments": args}, expected_func=fn)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 16 sandbox endpoints verified functional with HTTP 200 execution.")


### Stage 3: Load Model & Tokenizer (Fast Smoke Test: Qwen2.5-1.5B or 7B)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model Selection:
# Use Qwen2.5-1.5B-Instruct for ultra-fast 3-minute smoke testing
# Set to 'Qwen/Qwen2.5-7B-Instruct' whenever you want the 7B model
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
max_seq_length = 1536

print(f"[*] Loading {MODEL_ID} in 4-bit precision for rapid smoke test...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[✓] Model and tokenizer loaded successfully on device: {next(model.parameters()).device}")


### Stage 4: Base Model & Constrained Decoding Smoke Evaluation (25 Samples)


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from vericall.eval.evaluate import evaluate_predictions, apply_constrained_decoding

# Load unseen API test set (each item has prompt with 6 candidate schemas)
unseen_path = Path("data/eval_unseen_apis.jsonl")
all_test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Total frozen unseen API evaluation requests available: {len(all_test_records)}")

# Slice 25 samples for rapid smoke testing
test_records = all_test_records[:25]
print(f"[*] Running smoke evaluation on {len(test_records)} requests...")

assert "Available Tools:" in test_records[0]["prompt"], "Prompt missing candidate tool schemas!"
print(f"[✓] Verified in-prompt candidate schemas present. Target: {test_records[0]['target_function']}")

model.eval()
device = next(model.parameters()).device

base_predictions = []
print("\n[*] Generating Base Model predictions...")
for item in tqdm(test_records):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    base_predictions.append(resp)

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Smoke Accuracy: {base_results['accuracy_pct']}% ({base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             Errors: {base_results['error_taxonomy']}")

# Constrained decoding
print("\n[*] Evaluating Base Model + Constrained Schema Decoding...")
constrained_preds = [apply_constrained_decoding(pred, validator.registry) for pred in base_predictions]
constrained_results = evaluate_predictions(test_records, constrained_preds, "base_constrained")
print(f"[Constrained Decoding] Smoke Accuracy: {constrained_results['accuracy_pct']}% ({constrained_results['passed_requests']}/{constrained_results['total_requests']})")


### Stage 5: QLoRA Supervised Fine-Tuning (SFT) with Completion-Only Masking (15 Steps)


In [ ]:
import inspect
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Load 50 training records for fast smoke training
train_file = Path("data/train_sft.jsonl")
all_train_records = [json.loads(line) for line in open(train_file, "r", encoding="utf-8") if line.strip()]
smoke_train_records = all_train_records[:50]
print(f"[*] Loaded {len(smoke_train_records)} SFT training records for fast smoke training.")

train_dataset = Dataset.from_list([{"text": r["text"]} for r in smoke_train_records])

# RESILIENT DataCollatorForCompletionOnlyLM (Supports any TRL version)
response_template = "<|im_start|>assistant\n"

def get_completion_collator(response_template, tokenizer):
    try:
        from trl import DataCollatorForCompletionOnlyLM
        print("[✓] Loaded DataCollatorForCompletionOnlyLM from trl.")
        return DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)
    except (ImportError, AttributeError):
        print("[*] Using resilient custom DataCollatorForCompletionOnlyLM fallback...")
        class SafeCompletionCollator:
            def __init__(self, resp_tpl, tok):
                self.tok = tok
                self.resp_tpl = resp_tpl
                self.resp_ids = tok.encode(resp_tpl, add_special_tokens=False)

            def __call__(self, examples):
                texts = [e["text"] for e in examples]
                batch = self.tok(texts, padding=True, truncation=True, max_length=1536, return_tensors="pt")
                labels = batch["input_ids"].clone()
                n = len(self.resp_ids)
                for i in range(labels.shape[0]):
                    ids = batch["input_ids"][i].tolist()
                    found = -1
                    for j in range(len(ids) - n + 1):
                        if ids[j:j+n] == self.resp_ids:
                            found = j + n
                            break
                    if found != -1:
                        labels[i, :found] = -100
                    else:
                        labels[i, :] = -100
                batch["labels"] = labels
                return batch
        return SafeCompletionCollator(response_template, tokenizer)

collator = get_completion_collator(response_template, tokenizer)

# Ensure trainable parameters in float32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

try:
    from trl import SFTConfig
    sft_args = SFTConfig(
        output_dir="checkpoints/smoke_runs_sft",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        max_steps=15,  # 15 fast steps (~30s)
        learning_rate=2e-4,
        optim="adamw_torch",
        warmup_steps=3,
        logging_steps=5,
        seed=42,
        fp16=False,
        bf16=False,
        report_to="none",
        max_length=max_seq_length,
        dataset_text_field="text",
    )
except ImportError:
    from transformers import TrainingArguments
    sft_args = TrainingArguments(
        output_dir="checkpoints/smoke_runs_sft",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        max_steps=15,
        learning_rate=2e-4,
        optim="adamw_torch",
        warmup_steps=3,
        logging_steps=5,
        seed=42,
        fp16=False,
        bf16=False,
        report_to="none",
    )

sft_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "data_collator": collator,
    "args": sft_args,
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    sft_kwargs["processing_class"] = tokenizer
else:
    sft_kwargs["tokenizer"] = tokenizer

sft_trainer = SFTTrainer(**sft_kwargs)

print("[*] Starting fast SFT smoke training (15 steps)...")
sft_trainer.train()

model.save_pretrained("checkpoints/smoke_sft_adapter")
tokenizer.save_pretrained("checkpoints/smoke_sft_adapter")
print("[✓] SFT smoke training complete. Adapter saved to checkpoints/smoke_sft_adapter.")


### Stage 6: SFT Model Smoke Evaluation (25 Samples)


In [ ]:
model.eval()
device = next(model.parameters()).device

sft_predictions = []
print("\n[*] Generating predictions on 25 unseen requests with SFT Model...")
for item in tqdm(test_records):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    sft_predictions.append(resp)

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Smoke Accuracy: {sft_results['accuracy_pct']}% ({sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            Errors: {sft_results['error_taxonomy']}")


### Stage 7: Execution-Grounded Near-Miss DPO Mining (25 Pairs)


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=25)
print(f"[✓] Successfully loaded/mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Preference Pair:")
print("  Chosen:  ", dpo_pairs[0]["chosen"])
print("  Rejected:", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO) (10 Steps)


In [ ]:
import os
import sys
import types
import inspect
import torch

# 1. Mock optional deps
for pkg in ["llm_blender", "liger_kernel", "liger_kernel.transformers"]:
    if pkg not in sys.modules:
        sys.modules[pkg] = types.ModuleType(pkg)

import transformers.models.auto.modeling_auto
if not hasattr(transformers.models.auto.modeling_auto, "MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES"):
    transformers.models.auto.modeling_auto.MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES = {}

try:
    import trl.import_utils
    trl.import_utils.is_llmblender_available = lambda: False
    trl.import_utils.is_liger_kernel_available = lambda: False
except Exception:
    pass

# 2. Unload previously-patched DPO modules to ensure fresh import
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ["dpo_trainer", "dpo_config", "judges", "callbacks"]):
        del sys.modules[mod]

# 3. Bridge Trainer.__init__ (tokenizer -> processing_class)
import transformers.trainer
_trainer_sig = inspect.signature(transformers.trainer.Trainer.__init__).parameters
if "tokenizer" not in _trainer_sig and "processing_class" in _trainer_sig:
    if not getattr(transformers.trainer.Trainer.__init__, "_is_patched", False):
        _orig_trainer_init = transformers.trainer.Trainer.__init__
        def _patched_trainer_init(self, *args, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            return _orig_trainer_init(self, *args, **kwargs)
        _patched_trainer_init._is_patched = True
        transformers.trainer.Trainer.__init__ = _patched_trainer_init

# 4. Fresh import
from trl.trainer.dpo_trainer import DPOTrainer as _CleanDPOTrainer
from trl.trainer.dpo_config import DPOConfig
from datasets import Dataset

# 5. Robust subclass: adapts transformers >= 4.46 signatures
class RobustDPOTrainer(_CleanDPOTrainer):
    def get_batch_samples(self, epoch_iterator, num_batches, *args, **kwargs):
        batch_samples = []
        for _ in range(num_batches):
            try:
                batch_samples.append(next(epoch_iterator))
            except StopIteration:
                break
        num_items_in_batch = None
        if hasattr(self, "_get_num_items_in_batch"):
            try:
                device = args[0] if len(args) > 0 else kwargs.get("device", getattr(self.args, "device", None))
                num_items_in_batch = self._get_num_items_in_batch(batch_samples, device)
            except Exception:
                num_items_in_batch = None
        return batch_samples, num_items_in_batch

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        return _CleanDPOTrainer.compute_loss(self, model, inputs, return_outputs=return_outputs)

    def log(self, logs, *args, **kwargs):
        return _CleanDPOTrainer.log(self, logs)

# 6. DPO Config
cfg_params = inspect.signature(DPOConfig.__init__).parameters
dpo_cfg_kwargs = {
    "output_dir": "checkpoints/smoke_runs_dpo",
    "beta": 0.1,
    "learning_rate": 5e-6,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 2,
    "max_steps": 10,
    "optim": "adamw_torch",
    "warmup_steps": 2,
    "logging_steps": 2,
    "seed": 42,
    "fp16": False,
    "bf16": False,
    "report_to": "none",
    "remove_unused_columns": False,
    "max_length": 1536,
    "max_prompt_length": 1280,
    "save_strategy": "no",
}

if "rpo_alpha" in cfg_params:
    dpo_cfg_kwargs["rpo_alpha"] = 0.2
    print("[✓] Enabled RPO NLL anchor (rpo_alpha=0.2) to prevent policy collapse.")

dpo_config = DPOConfig(**dpo_cfg_kwargs)

dpo_train_dataset = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in dpo_pairs
])

# 7. Cast trainable parameters to float32 & enable input grads
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

# 8. Train
dpo_trainer = RobustDPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train_dataset,
    tokenizer=tokenizer,
)

print("[*] Starting fast QLoRA DPO alignment (10 steps)...")
dpo_trainer.train()

model.save_pretrained("checkpoints/smoke_dpo_adapter")
tokenizer.save_pretrained("checkpoints/smoke_dpo_adapter")
print("[✓] DPO smoke training complete. Adapter saved to checkpoints/smoke_dpo_adapter.")


### Stage 9: Final Smoke Evaluation & McNemar Tests (25 Samples)


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

model.eval()
device = next(model.parameters()).device

dpo_predictions = []
print("\n[*] Generating final predictions with VeriCall (SFT+DPO) on 25 unseen requests...")
for item in tqdm(test_records):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    dpo_predictions.append(resp)

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# Compute McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"VERICALL FAST SMOKE TEST RESULTS (N={len(test_records)} Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  Passed: {base_results['passed_requests']}/{base_results['total_requests']}")
print(f"[Base + Constr]  Accuracy: {constrained_results['accuracy_pct']}%  Passed: {constrained_results['passed_requests']}/{constrained_results['total_requests']}")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  Passed: {sft_results['passed_requests']}/{sft_results['total_requests']}")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  Passed: {dpo_results['passed_requests']}/{dpo_results['total_requests']}")
print(f"=======================================================")


### Stage 10: Save Smoke Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
import json
import os
import shutil

summary_data = {
    "metadata": {
        "model_base": MODEL_ID,
        "eval_set": "eval_unseen_apis_smoke.jsonl",
        "eval_count": len(test_records),
        "mode": "smoke_test_run",
    },
    "configs": {
        "base_model": base_results,
        "base_constrained": constrained_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/smoke_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print("[✓] Saved smoke evaluation summary to results/smoke_eval_summary.json")

# Copy to /kaggle/working for direct download
if os.path.exists("/kaggle/working"):
    shutil.copy2("results/smoke_eval_summary.json", "/kaggle/working/smoke_eval_summary.json")
    print("[✓] Copied to /kaggle/working/smoke_eval_summary.json for direct artifact download")

print("\n[✓✓✓] FAST SMOKE TEST FINISHED SUCCESSFULLY IN ~4 MINUTES!")
